# Build the shared volume cache

Preprocessing is repeated every epoch for no benefit: reading and resampling a
139 MB FLAIR costs ~580 ms per sample and always yields the same 128x128x80
volume. Cached, a sample loads in ~14 ms.

The cache does not depend on the fold -- folds only change which subjects are
training and which are validation -- so build it **once**, save the output as a
Kaggle Dataset, and attach it to the training notebook for all five folds.

**Accelerator: None.** This is CPU work; do not spend GPU quota on it.


## 1. Setup

In [ ]:
# Build the shared volume cache. Run this ONCE, then save the output as a
# Kaggle Dataset and attach it to the training notebook for every fold.
#
# No GPU is needed here -- set Accelerator to None and save the quota.

import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Payz111/mri-stroke-assistance.git"
REPO_DIR = Path("/kaggle/working/mri-stroke-assist")

if REPO_DIR.exists():
    print(f"Removing previous checkout at {REPO_DIR}")
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[name]
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "monai", "nibabel", "SimpleITK", "pyyaml"],
    check=True,
)

commit = subprocess.run(
    ["git", "log", "--oneline", "-1"], capture_output=True, text=True, cwd=REPO_DIR
).stdout.strip()
print(f"Checked out: {commit}")


## 2. Locate the datasets

In [ ]:
from src.data.kaggle_paths import locate_isles, locate_soop

SOOP_ROOT = locate_soop()
isles_root, isles_derivatives = locate_isles()

print(f"ISLES root       : {isles_root}")
print(f"ISLES derivatives: {isles_derivatives}")
print(f"SOOP root        : {SOOP_ROOT}")


## 3. Check both sources resolve

In [ ]:
# Smoke run on a handful of subjects first: a typo in a path is far cheaper to
# find now than 40 minutes into packing 12 GB.
from src.data.isles22_dataset import ISLES22Dataset
from src.data.soop_dataset import SOOPDataset
from src.data.volume_cache import build_pack

isles_all = ISLES22Dataset(
    data_root=isles_root,
    derivatives_root=isles_derivatives,
    split_file=None,
)
soop_all = SOOPDataset(data_root=SOOP_ROOT, require_mask=True)

print(f"ISLES subjects: {len(isles_all)}")
print(f"SOOP subjects : {len(soop_all)}")
print(f"TOTAL         : {len(isles_all) + len(soop_all)}")

if len(soop_all) == 0:
    raise RuntimeError(
        "SOOP resolved to 0 subjects. Building the cache now would produce an "
        "ISLES-only pack and every fold would silently train on it."
    )


## 4. Build the pack

In [ ]:
import time
from pathlib import Path

CACHE_DIR = Path("/kaggle/working/stroke_cache")

started = time.time()
index = build_pack([isles_all, soop_all], CACHE_DIR, spatial_size=(128, 128, 80), log_every=100)
elapsed = time.time() - started

print(f"\nBuilt in {elapsed / 60:.1f} min")
print(f"  packed : {index['n_subjects']}")
print(f"  skipped: {len(index['skipped'])} {index['skipped'][:5]}")
for path in sorted(CACHE_DIR.iterdir()):
    print(f"  {path.name:12s} {path.stat().st_size / 1e9:6.2f} GB")


## 5. Verify before saving

In [ ]:
# Read a few rows back and confirm the pack is usable before saving 12 GB.
from src.data.volume_cache import PackedVolumeDataset

check = PackedVolumeDataset(CACHE_DIR, augment=False)
sample = check[0]

print(f"subjects in cache: {len(check)}")
print(f"image shape      : {tuple(sample['image'].shape)}  dtype {sample['image'].dtype}")
print(f"label shape      : {tuple(sample['label'].shape)}")
print(f"label is binary  : {set(sample['label'].unique().tolist()) <= {0.0, 1.0}}")
print(f"first subject    : {sample['subject_id']}")

# Every fold's validation split must be fully present, or training would run on
# a quietly shortened set.
import json

for fold in range(5):
    split = json.loads(Path(f"{REPO_DIR}/data/splits/fold_{fold}.json").read_text())
    PackedVolumeDataset(CACHE_DIR, split["train"] + split["val"])
    print(f"fold {fold}: all {len(split['train']) + len(split['val'])} ISLES subjects present")

print("\nCache is complete. Save this notebook's output as a Kaggle Dataset,")
print("then attach it to 05_kaggle_attention_training for every fold.")
